In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

565

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
index = build_index(documents)

doc_idx = {doc["id"]: doc for doc in documents}

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [5]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join now. If you want a certificate, though, you need to submit your project while submissions are still being accepted.'

In [6]:
assistant.total_cost()

0.00052725

In [7]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    return {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

In [8]:
assistant.reset_usage()

In [9]:
from concurrent.futures import ThreadPoolExecutor

from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=3) as pool:
    answers = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [10]:
assistant.total_cost()

0.6112829999999998

In [11]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)
df_answers.head()

,question,answer_llm,answer_orig,document
0,I just found this course late — can I still jo...,"Yes, you can still join now. You’re accepted, ...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I enroll after the course started, am I sti...","Yes, if you enroll after the course has starte...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,What do I need to do to qualify for a certific...,You need to pass the Capstone project to get t...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Is there a deadline for project submission if ...,"Yes. If you want a certificate, you need to su...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Can I still take the course now, and will a la...",Yes — you can still take the course now.\n\nA ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
